<a href="https://colab.research.google.com/github/thornton330j/IS-4487/blob/main/Assignments/assignment_11_ThorntonJackson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [ ]:
# 🔧 Add code here
import pandas as pd
import requests

# Define
github_csv_url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/airbnb_listings.csv'
airbnb_csv_path = 'airbnb_listings.csv'

# Load dataset
df = pd.read_csv(airbnb_csv_path)

display(df.head())
df.info()
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 77 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            459 non-null    int64  
 1   listing_url                                   459 non-null    object 
 2   scrape_id                                     459 non-null    int64  
 3   last_scraped                                  459 non-null    object 
 4   source                                        459 non-null    object 
 5   name                                          459 non-null    object 
 6   description                                   449 non-null    object 
 7   neighborhood_overview                         196 non-null    object 
 8   host_url                                      459 non-null    object 
 9   host_name                                     459 non-null    obj

### ✍️ Your Response: 🔧
1.  The dataset includes listing information such as host attributes, listing details, location data, property characteristics, and calendar/booking variables.
2. The dataset contains 459 rows and 77 columns.



## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [ ]:
# 🔧 Add code here
# List of columns to drop
columns_to_drop = [
    'id', 'listing_url', 'scrape_id', 'last_scraped', 'name',
    'description', 'neighborhood_overview', 'host_url', 'host_name',
    'host_since', 'host_location', 'host_about', 'host_thumbnail_url',
    'host_picture_url', 'host_neighbourhood', 'host_verifications',
    'neighbourhood', 'bathrooms_text', 'amenities', 'calendar_updated',
    'license', 'first_review', 'last_review', 'calendar_last_scraped',
    'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights',
    'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm'
]
df.info()
print(f"\nUpdated Rows: {df.shape[0]}, Columns: {df.shape[1]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 47 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   source                                        459 non-null    object 
 1   host_response_time                            431 non-null    object 
 2   host_response_rate                            431 non-null    object 
 3   host_acceptance_rate                          446 non-null    object 
 4   host_is_superhost                             452 non-null    object 
 5   host_listings_count                           459 non-null    int64  
 6   host_total_listings_count                     459 non-null    int64  
 7   host_has_profile_pic                          459 non-null    object 
 8   host_identity_verified                        459 non-null    object 
 9   neighbourhood_cleansed                        459 non-null    obj

### ✍️ Your Response: 🔧
1.  I dropped 'id', 'listing_url', 'scrape_id', 'last_scraped', 'name',
    'description', 'neighborhood_overview', 'host_url', 'host_name',
    'host_since', 'host_location', 'host_about', 'host_thumbnail_url',
    'host_picture_url', 'host_neighbourhood', 'host_verifications',
    'neighbourhood', 'bathrooms_text', 'amenities', 'calendar_updated',
    'license', 'first_review', 'last_review', 'calendar_last_scraped',
    'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights',
    'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm'.
  I dropped them because they added noise and may not help to predict price.

2.  Increased noise and overfitting, high dimensionality, multicollinearity, and data leakage could all occur if these variables were included.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [ ]:
# 🔧 Add code here
# Calculate the correlation matrix
correlation_matrix = df.select_dtypes(include=['number']).corr()

# Get correlations with the price column
price_correlations = correlation_matrix['price'].sort_values(ascending=False)
display(price_correlations)

# Exclude price to find other top predictors
top_correlated_features = price_correlations.drop('price').abs().sort_values(ascending=False)
display(top_correlated_features.head(10))

,price
price,1.000000
accommodates,0.579588
beds,0.547032
bedrooms,0.499286
bathrooms,0.468030
estimated_revenue_l365d,0.249488
availability_30,0.108409
availability_60,0.060509
availability_90,0.040997
calculated_host_listings_count_entire_homes,0.033206


,price
accommodates,0.579588
beds,0.547032
bedrooms,0.499286
bathrooms,0.468030
estimated_revenue_l365d,0.249488
review_scores_communication,0.131798
longitude,0.118913
availability_30,0.108409
review_scores_checkin,0.103223
reviews_per_month,0.090843


### ✍️ Your Response: 🔧
1. Variables with the strongest positive correlations were accommodates (0.579588), beds (0.547032), bedrooms (0.499286), and bathrooms (0.468030). Variables with the strongest negative correlations were review_scores_communication (-0.131798), longitude (-0.118913), and review_scores_checkin (-0.104278).
2. The variables that show the strongest relationship with price and are likely useful predictors are accommodates, beds, bedrooms, and bathrooms.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [ ]:
# 🔧 Add code here
y = df['price']
X = df.drop('price', axis=1)

print(f"Target variable 'y' shape: {y.shape}")
print(f"Features 'X' shape: {X.shape}")

Target variable 'y' shape: (459,)
Features 'X' shape: (459, 46)


### ✍️ Your Response: 🔧
1. I am using all columns in the dataframe except for price as features.
2.  This is a regression problem because the target variable is a continuous numerical variable. We are trying to predict a specific numerical value rather than classifying it into a predefined set of categories.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [ ]:
# 🔧 Add code here
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (367, 46)
X_test shape: (92, 46)
y_train shape: (367,)
y_test shape: (92,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [39]:
# 🔧 Add code here
# Handle percentage strings in host_response_rate and host_acceptance_rate
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in X_train.columns and X_train[col].dtype == 'object': # Check dtype before .str accessor
        X_train[col] = X_train[col].str.replace('%', '', regex=False).astype(float) / 100
    if col in X_test.columns and X_test[col].dtype == 'object': # Check dtype before .str accessor
        X_test[col] = X_test[col].str.replace('%', '', regex=False).astype(float) / 100

# Map 't'/'f' to 1/0 for boolean-like columns
tf_cols = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'has_availability']
for col in tf_cols:
    if col in X_train.columns and X_train[col].dtype == 'object': # Check dtype before .map
        X_train[col] = X_train[col].map({'t': 1, 'f': 0})
    if col in X_test.columns and X_test[col].dtype == 'object': # Check dtype before .map
        X_test[col] = X_test[col].map({'t': 1, 'f': 0})

# Impute missing numerical values with the median from the training set
median_imputation_values = {}
for col in X_train.select_dtypes(include=np.number).columns:
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        median_imputation_values[col] = median_val
        X_train[col] = X_train[col].fillna(median_val)
        if col in X_test.columns:
            X_test[col] = X_test[col].fillna(median_val)

# Identify categorical columns (object dtype after initial conversions)
categorical_cols = X_train.select_dtypes(include='object').columns

# Apply one-hot encoding
X_train_dummies = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_dummies = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - this is crucial if the test set has categories not present in train or vice-versa
X_test_dummies = X_test_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

# Initialize and fit the Linear Regression model
model = LinearRegression()
model.fit(X_train_dummies, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_dummies)

print("Categorical columns in X_test have been one-hot encoded and aligned with X_train.")
print(f"New shape of X_test: {X_test_dummies.shape}")
print("Linear Regression model fitted and predictions made.")

Categorical columns in X_test have been one-hot encoded and aligned with X_train.
New shape of X_test: (92, 78)
Linear Regression model fitted and predictions made.


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [40]:
# 🔧 Add code here
# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)

# Calculate R-squared score
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

Mean Squared Error (MSE): 4606.32
R-squared (R²): 0.18


### ✍️ Your Response: 🔧
1. My R² score is 0.18. This means the model explains approximately 18% of the variance in Airbnb prices. This is a low score, indicating that a significant portion of the price variation is not accounted for,

2. My MSE is 4606.32. This is large. To improve it, I could consider feature engineering, feature selection, handling outliers, or trying different models.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [41]:
# 🔧 Add code here
# Create a DataFrame of coefficients
coefficients_df = pd.DataFrame({
    'Feature': X_train_dummies.columns,
    'Coefficient': model.coef_
})

# Sort by the absolute value of the coefficients to see the most impactful features
coefficients_df['Abs_Coefficient'] = coefficients_df['Coefficient'].abs()
coefficients_df = coefficients_df.sort_values(by='Abs_Coefficient', ascending=False).drop(columns='Abs_Coefficient')

display(coefficients_df.head(20))


,Feature,Coefficient
1,host_acceptance_rate,3.777155e+07
0,host_response_rate,1.809082e+07
4,latitude,-9.199177e+02
5,longitude,4.578000e+02
40,host_response_time_within an hour,-1.831422e+02
37,host_response_time_missing_category,-1.624152e+02
39,host_response_time_within a few hours,-1.616563e+02
63,property_type_Entire place,-1.613203e+02
27,review_scores_communication,-1.494022e+02
29,review_scores_value,1.320942e+02


### ✍️ Your Response: 🔧
1.  Based on the coefficients, the features with the largest positive impact on price were host_acceptance_rate, host_response_rate and longitude.
2.  Several features had surprisingly negative coefficients such as latitude, host_response_time_within an hour, property_type_Entire place, review_scores_communication, and review_scores_checkin.
3.  Host responsiveness suggests that host acceptance and response rates are highly associated with price. Geographical importance reinforces the intuitive understanding that location is a primary driver of property value.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [ ]:
# Add code here 🔧

### ✍️ Your Response: 🔧
1.

2.

3.

4.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1.

2.

3.

4.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_ThorntonJackson.ipynb"